# AI Engineering Interview Prep
## Section: FastAPI

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 551+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## ⚡ Section 18 — FastAPI

### Q1. Explain how to implement custom request validation and serialization beyond Pydantic defaults.
**A:** Pydantic handles most validation, but you can write custom validation rules using decorators like `@validator` (or `@field_validator` in Pydantic v2).

```python
from pydantic import BaseModel, validator

class User(BaseModel):
    username: str
    password: str

    @validator("password")
    def password_length(cls, value):
        if len(value) < 8:
            raise ValueError("Password must be at least 8 characters long")
        return value
```

For custom serialization, you can modify data in Pydantic using custom encoders or by transforming the dict before returning it, or by overriding the `response_model` serialization rules. Custom validation is useful when business rules require complex validation logic, input data must be transformed before processing, or security checks must be enforced.

### Q2. How do you design FastAPI microservices and handle inter-service communication?
**A:** FastAPI is commonly used to build microservices where each service handles a specific business function. Design considerations include independent service deployment, separate databases, clear API contracts, and centralized authentication and logging.

Communication methods include:
1. **REST APIs (HTTP):** Services communicate using HTTP endpoints (e.g. using `httpx`).
2. **Message Queues:** Asynchronous event-driven communication using tools like RabbitMQ, Kafka, or Redis.
3. **Service Discovery:** Allows services to locate each other dynamically (e.g. Consul or Kubernetes DNS).

### Q3. What strategies do you use for caching in FastAPI applications?
**A:** Caching reduces database/API calls, saving infrastructure costs. Strategies include:
1. **In-Memory Caching:** Using Python dicts or libraries like `cachetools` (for small, transient data).
2. **Redis Caching:** Best for production/distributed setups. Stores data in an external Redis instance.

```python
import aioredis

redis = await aioredis.from_url("redis://localhost")

async def get_cached_data(key):
    return await redis.get(key)
```

3. **Response Caching:** Cache the entire HTTP response (using middleware or decorators) to serve repeat requests instantly.

### Q4. How do you deploy a FastAPI application with Docker and Uvicorn/Gunicorn?
**A:** FastAPI applications are typically deployed using ASGI servers such as Uvicorn or Gunicorn combined with Uvicorn workers. Docker containerizes the application for consistency.

Example Dockerfile:
```dockerfile
FROM python:3.10
WORKDIR /app
COPY . /app
RUN pip install -r requirements.txt
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "80"]
```

For production, Gunicorn is run with Uvicorn workers for process management and scaling:
```bash
gunicorn main:app -k uvicorn.workers.UvicornWorker
```

### Q5. Explain lifespan events (startup/shutdown) in FastAPI and their use cases.
**A:** Lifespan events manage resource setups and cleanups during the application lifecycle. Using `@asynccontextmanager` (replaces legacy `@app.on_event` startup/shutdown handlers):

```python
from fastapi import FastAPI
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup logic: open db pool, load ML model
    print("Application started")
    yield
    # Shutdown logic: close db pool, free GPU memory
    print("Application stopped")

app = FastAPI(lifespan=lifespan)
```

### Q6. How do you implement rate limiting and request throttling in FastAPI?
**A:** Rate limiting protects APIs from abuse and DDoS. Since FastAPI doesn't have it built-in, we use middleware or libraries like `SlowAPI` (which wraps `limits` and integrates with Redis/memory stores):

```python
from slowapi import Limiter
from slowapi.util import get_remote_address
from fastapi import Request

limiter = Limiter(key_func=get_remote_address)

@app.get("/items")
@limiter.limit("5/minute")
async def get_items(request: Request):
    return {"message": "Rate limited endpoint"}
```

Common strategies include limiting requests per IP, per authenticated user, or using Redis to track request counts.

### Q7. What is middleware in FastAPI and how do you create custom middleware?
**A:** Middleware is code that runs before or after every request. It is used for logging, authentication checks, request tracking, and response modification. Created via the `@app.middleware("http")` decorator:

```python
from fastapi import Request
import time

@app.middleware("http")
async def log_request_time(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)
    return response
```

### Q8. How do you write tests for FastAPI applications using TestClient?
**A:** TestClient simulates HTTP requests without running the server. It wraps the `requests` library:

```python
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def test_read_root():
    response = client.get("/")
    assert response.status_code == 200
    assert response.json() == {"message": "Hello World"}
```

### Q9. How do you implement CORS (Cross-Origin Resource Sharing) in FastAPI?
**A:** CORS allows a frontend application running on one domain to access an API hosted on another domain. FastAPI supports CORS using `CORSMiddleware`:

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # In production, list exact domains
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
```

### Q10. How do you connect FastAPI with a database using SQLAlchemy?
**A:** Set up a database session dependency and inject it into endpoints:

```python
from sqlalchemy.orm import Session
from fastapi import Depends

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.get("/users/")
def get_users(db: Session = Depends(get_db)):
    return db.query(User).all()
```

### Q11. How do you handle errors and exceptions in FastAPI? Explain HTTPException.
**A:** FastAPI provides `HTTPException` to raise standard HTTP error codes, which are serialized to JSON:

```python
from fastapi import HTTPException

@app.get("/items/{item_id}")
def read_item(item_id: int):
    if item_id != 1:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"item_id": item_id}
```

You can also register custom exception handlers via `@app.exception_handler()` for global error management.

### Q12. What is Dependency Injection in FastAPI, and how does it work?
**A:** Dependency Injection lets you share and reuse logic (like db sessions, auth guards, parameters) across routes. You use `Depends()`:

```python
from fastapi import Depends

def common_parameters():
    return {"limit": 10}

@app.get("/items/")
def read_items(params: dict = Depends(common_parameters)):
    return params
```
FastAPI executes `common_parameters` and injects the returned value into `params` before executing the endpoint.

### Q13. How do you handle database migrations in FastAPI using Alembic?
**A:** Alembic is a database migration tool used with SQLAlchemy to manage schema changes without losing data.

Process:
1. Initialize: `alembic init alembic`
2. Configure `env.py` with SQLAlchemy's metadata and database URL.
3. Auto-generate a migration script: `alembic revision --autogenerate -m "Create tables"`
4. Apply to the database: `alembic upgrade head`

### Q14. Explain APIRouter and how to structure a large FastAPI application.
**A:** `APIRouter` splits route mappings into separate files/modules to keep the codebase clean and modular:

```python
# routers/items.py
from fastapi import APIRouter
router = APIRouter()
@router.get("/items")
def get_items(): return {"items": []}

# main.py
app.include_router(items.router)
```
Large FastAPI applications are usually structured with directories for `routers/`, `models/`, `schemas/`, `database/`, and `services/`.

### Q15. How do you implement WebSockets in FastAPI for real-time communication?
**A:** WebSockets allow continuous, bidirectional communication between client and server over a single TCP connection:

```python
from fastapi import WebSocket

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    while True:
        message = await websocket.receive_text()
        await websocket.send_text(f"Message received: {message}")
```

### Q16. What are Background Tasks in FastAPI, and when should you use them?
**A:** `BackgroundTasks` executes operations *after* sending the response, keeping client latency low. Useful for sending emails, logging events, or parsing reports:

```python
from fastapi import BackgroundTasks

def write_log(message: str):
    with open("log.txt", "a") as file:
        file.write(message)

@app.post("/notify/")
def send_notification(background_tasks: BackgroundTasks):
    background_tasks.add_task(write_log, "Notification sent")
    return {"message": "Task scheduled"}
```

### Q17. How do you implement OAuth2 with JWT authentication in FastAPI?
**A:** You generate a JSON Web Token on successful login and verify it on subsequent calls using dependencies:

```python
from fastapi.security import OAuth2PasswordBearer
from fastapi import Depends

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

@app.get("/users/me")
async def read_users_me(token: str = Depends(oauth2_scheme)):
    return {"token": token}
```
In production, JWTs store encoded user data, which are validated using a secret key and check expiration time to prevent unauthorized access.

### Q18. Explain async and await in FastAPI. When should you use async def vs def?
**A:** FastAPI is built on ASGI and supports non-blocking concurrency:
- **`async def`:** Use when the function waits for I/O (database, external API calls, file operations) using `await`. It runs on the main event loop, yielding control while waiting.
- **`def`:** Use for CPU-bound tasks or standard blocking operations. FastAPI offloads sync `def` functions to an external thread pool to prevent blocking the event loop.

### Q19. What is the purpose of response_model in FastAPI path operations?
**A:** `response_model` defines the output format, filtering and serializing values. It hides sensitive/unnecessary attributes and structures public API data:

```python
class ItemResponse(BaseModel):
    name: str
    price: float

@app.get("/items/{item_id}", response_model=ItemResponse)
def get_item(item_id: int):
    return {"name": "Laptop", "price": 50000, "internal_code": "XYZ"}
# Output excludes "internal_code"
```

### Q20. How do you handle form data and file uploads in FastAPI?
**A:** FastAPI handles multi-part requests using specific parameters:
- **Form Data:** Received using the `Form` class.
- **File Uploads:** Received using `UploadFile` or `File` classes.

```python
from fastapi import Form, UploadFile, File

@app.post("/login/")
def login(username: str = Form(...), file: UploadFile = File(...)):
    return {"username": username, "filename": file.filename}
```

### Q21. Explain the different HTTP methods and their usage in FastAPI.
**A:** HTTP methods define the action to perform on a resource:
- **GET:** Retrieve records or resources. `@app.get()`
- **POST:** Create new records or resources. `@app.post()`
- **PUT:** Replace/update an entire resource. `@app.put()`
- **PATCH:** Partially update specific fields. `@app.patch()`
- **DELETE:** Remove a resource. `@app.delete()`

### Q22. How do you define request body using Pydantic models?
**A:** Define a Pydantic class inheriting from `BaseModel`. Passing it as a route parameter registers it as the request body:

```python
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool

@app.post("/items/")
def create_item(item: Item):
    return item
```
FastAPI automatically parses and validates incoming payloads, returning a validation error if the formatting is incorrect.

### Q23. What are path parameters and query parameters in FastAPI? How do you define them?
**A:** FastAPI receives parameters from the URL:
- **Path Parameters:** Part of the URL route path. Used to identify a specific resource:
  ```python
  @app.get("/items/{item_id}")
  def read_item(item_id: int): ...
  ```
- **Query Parameters:** Added after `?` in the URL (typically used for filtering or optional inputs):
  ```python
  @app.get("/items/")
  def read_items(limit: int = 10): ...
  ```

### Q24. How do you create a basic FastAPI application with a GET endpoint?
**A:** Import the `FastAPI` class, instantiate it, and define a route function decorated with `@app.get()`:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "Hello World"}
```

### Q25. What is FastAPI, and what are its key features?
**A:** FastAPI is a modern, high-performance, ASGI web framework for building APIs with Python 3.8+. Key features include:
- **Performance:** Extremely fast, on par with NodeJS and Go (using Starlette and Uvicorn).
- **Asynchronous:** Native support for `async`/`await` request handling.
- **Data Validation:** Automatic validation using Python type hints and Pydantic.
- **Automatic Documentation:** Instantly generates interactive Swagger UI and ReDoc interfaces.

### Q26. Compare FastAPI with Flask and Django REST Framework.
**A:** - **FastAPI:** Async-first, high performance, automatically validates data via Pydantic, best for microservices and real-time/streaming applications.
- **Flask:** Lightweight, flexible micro-framework. Synchronous by default, requiring external libraries for validation and docs.
- **Django REST Framework:** Built on Django's full-stack architecture. Includes ORM, authentication, and admin panel, but is heavier and synchronous.

### Q27. How does FastAPI automatically generate OpenAPI (Swagger) documentation?
**A:** FastAPI inspects route decorators, path parameters, type annotations, and Pydantic schemas. It compiles this metadata into a standard JSON schema that complies with the OpenAPI specification. It exposes this schema via `/docs` (interactive Swagger UI) and `/redoc` (structured documentation page) in real-time.

### Q28. What is Pydantic, and why is it integral to FastAPI?
**A:** Pydantic is a parsing and validation library that uses Python type hints. FastAPI uses it to validate query arguments, path parameters, and request payloads at runtime. Pydantic also serializes response models to clean JSON, filtering out fields not defined in the output schema.

### Q29. What is Starlette, and how does FastAPI build upon it?
**A:** Starlette is a lightweight ASGI toolkit/framework. It handles routing, WebSocket endpoints, middleware pipelines, and background tasks. FastAPI extends Starlette by adding type-hinted data validation, structured dependency injection, and automatic OpenAPI schema generation.

### Q30. Explain the difference between ASGI and WSGI. Why does FastAPI use ASGI?
**A:** - **WSGI:** Synchronous gateway protocol. Spawns threads/processes per request. Inefficient for WebSockets or async long-polling.
- **ASGI:** Asynchronous gateway protocol. Handles multiple concurrent connections in a single thread using an event loop.
FastAPI uses ASGI (via servers like Uvicorn) to support WebSockets, async token streaming, and massive request concurrency.